In [1]:
# Install required packages
!pip install fuzzywuzzy python-Levenshtein pandas -q

import pandas as pd
from fuzzywuzzy import fuzz, process
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Generate test data
def generate_test_data(num_invoices=20):
    companies = [
        "Saudi Telecom Company", "Al Rajhi Bank", "SABIC", "STC Solutions",
        "Almarai Company", "Jarir Bookstore", "BinDawood Group", "Saudi Electricity",
        "Mobily", "Zain Saudi", "NCB Bank", "Samba Financial", "Riyad Bank"
    ]
    vat_numbers = [f"3{random.randint(10000000000000, 99999999999999)}" for _ in range(len(companies))]
    invoices = []
    for i in range(num_invoices):
        original_idx = random.randint(0, len(companies)-1)
        original_name = companies[original_idx]
        original_vat = vat_numbers[original_idx]

        # Name errors (30% chance)
        if random.random() < 0.3:
            if random.random() < 0.5:
                name = original_name.replace("a", "e").replace("i", "y")
                if len(name) > 8:
                    name = name[:random.randint(8, len(name))]
            else:
                if len(original_name) > 3:
                    name = original_name[:-2] + "??"
                else:
                    name = original_name + "??"
        else:
            name = original_name

        # VAT errors (20% chance)
        if random.random() < 0.2:
            error_type = random.choice(['wrong', 'space', 'short'])
            if error_type == 'wrong':
                vat = original_vat[:-2] + str(random.randint(10, 99))
            elif error_type == 'space':
                vat = original_vat[:5] + " " + original_vat[5:10] + " " + original_vat[10:]
            else:
                vat = original_vat[:-3] if len(original_vat) > 10 else original_vat
        else:
            vat = original_vat

        invoices.append({
            'invoice_number': f"INV-{2025001 + i}",
            'supplier_name': name,
            'supplier_vat': vat,
            'invoice_date': (datetime.now() - timedelta(days=random.randint(0, 90))).strftime('%Y-%m-%d'),
            'amount': round(random.uniform(100, 50000), 2),
        })
    return pd.DataFrame(invoices), companies, vat_numbers

# Validation functions
def is_valid_vat_format(vat):
    clean = str(vat).replace(" ", "").replace("-", "").strip()
    return len(clean) == 15 and clean.isdigit() and clean.startswith('3')

def is_valid_zatca_date(date_str):
    try:
        invoice_date = pd.to_datetime(date_str)
        today = datetime.now()
        days_diff = (today - invoice_date).days
        return 0 <= days_diff <= 90
    except:
        return False

def validate_invoices(invoice_df, master_companies, master_vats):
    results = []
    for idx, row in invoice_df.iterrows():
        supplier = str(row['supplier_name'])
        vat = str(row['supplier_vat'])

        matched_name = "NOT FOUND"
        score = 0
        expected_vat = "N/A"

        # Date validation
        if 'invoice_date' in row and pd.notna(row['invoice_date']):
            if not is_valid_zatca_date(row['invoice_date']):
                results.append({
                    'invoice_no': row.get('invoice_number', f"Row {idx+1}"),
                    'supplier_provided': supplier,
                    'supplier_vat_provided': vat,
                    'best_match': "N/A",
                    'match_score': 0,
                    'expected_vat': "N/A",
                    'validation_status': "❌ FAIL - Invoice date outside ZATCA window (90 days)",
                })
                continue

        # VAT format validation
        if not is_valid_vat_format(vat):
            results.append({
                'invoice_no': row.get('invoice_number', f"Row {idx+1}"),
                'supplier_provided': supplier,
                'supplier_vat_provided': vat,
                'best_match': "N/A",
                'match_score': 0,
                'expected_vat': "N/A",
                'validation_status': "❌ FAIL - Invalid VAT format (must be 15 digits starting with 3)",
            })
            continue

        # Fuzzy match company name
        best_match = process.extractOne(supplier, master_companies, scorer=fuzz.token_sort_ratio)

        if best_match:
            matched_name, score = best_match
            matched_idx = master_companies.index(matched_name)
            expected_vat = master_vats[matched_idx]

            clean_vat = vat.replace(" ", "").replace("-", "").strip()
            clean_expected = expected_vat.replace(" ", "").replace("-", "").strip()
            vat_match = (clean_vat == clean_expected)

            if score >= 80 and vat_match:
                status = "✅ PASS - Valid invoice"
            elif score >= 80 and not vat_match:
                status = "⚠️ WARNING - Company name matches but VAT number mismatch"
            elif score < 80 and vat_match:
                status = "⚠️ WARNING - VAT matches but company name unrecognized (manual review)"
            else:
                status = "❌ FAIL - Neither name nor VAT matches master records"
        else:
            status = "❌ FAIL - Supplier not found in master database"

        results.append({
            'invoice_no': row.get('invoice_number', f"Row {idx+1}"),
            'supplier_provided': supplier,
            'supplier_vat_provided': vat,
            'best_match': matched_name,
            'match_score': score,
            'expected_vat': expected_vat,
            'validation_status': status,
        })
    return pd.DataFrame(results)

# Run validation
print("=" * 60)
print("ZATCA INVOICE VALIDATOR")
print("=" * 60)

print("\n📄 Generating test invoices...")
invoices, master_names, master_vats = generate_test_data(20)

print("\n📋 Sample invoices (first 5):")
print(invoices[['invoice_number', 'supplier_name', 'supplier_vat', 'invoice_date']].head())

print("\n🔍 Validating...")
results = validate_invoices(invoices, master_names, master_vats)

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(results[['invoice_no', 'supplier_provided', 'best_match', 'match_score', 'validation_status']].to_string(index=False))

# Summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
pass_count = len(results[results['validation_status'].str.contains("PASS")])
warn_count = len(results[results['validation_status'].str.contains("WARNING")])
fail_count = len(results[results['validation_status'].str.contains("FAIL")])

print(f"✅ PASS:     {pass_count}")
print(f"⚠️ WARNING: {warn_count}")
print(f"❌ FAIL:    {fail_count}")

# Export
results.to_csv("zatca_validation_results.csv", index=False)
invoices.to_csv("test_invoices.csv", index=False)
print("\n💾 Results saved to 'zatca_validation_results.csv' and 'test_invoices.csv'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.1 MB/s eta 0:00:00
ZATCA INVOICE VALIDATOR

📄 Generating test invoices...

📋 Sample invoices (first 5):
  invoice_number          supplier_name       supplier_vat invoice_date
0    INV-2025001        Almarai Company  33370 93863 93883   2026-05-16
1    INV-2025002             Zain Saudi    384537923841100   2026-02-28
2    INV-2025003        Jarir Bookstore    370064886056672   2026-04-30
3    INV-2025004  Saudi Telecom Company    372234356443130   2026-05-19
4    INV-2025005                  SAB??    360988643188228   2026-04-10

🔍 Validating...

RESULTS
 invoice_no     supplier_provided            best_match  match_score                                                      validation_status
INV-2025001       Almarai Company       Almarai Company          100                                                 ✅ PASS - Valid invoice
INV-2025002         